# Axon phase0b GPU resume

Attach `axon_phase0b_kaggle_bundle.zip` as a Kaggle dataset, enable GPU, then run the cells in order.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

work = Path('/kaggle/working/axon_phase0b')
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True, exist_ok=True)

bundle_candidates = (
    sorted(Path('/kaggle/input').glob('**/axon_phase0b_kaggle_bundle.axonbundle'))
    + sorted(Path('/kaggle/input').glob('**/axon_phase0b_kaggle_bundle.zip'))
)
if bundle_candidates:
    bundle = bundle_candidates[0]
    print('bundle zip:', bundle)
    with zipfile.ZipFile(bundle) as zf:
        zf.extractall(work)
else:
    local = Path('/kaggle/working/axon_phase0b_kaggle_bundle.axonbundle')
    if not local.exists():
        local = Path('/kaggle/working/axon_phase0b_kaggle_bundle.zip')
    if local.exists():
        print('local bundle zip:', local)
        with zipfile.ZipFile(local) as zf:
            zf.extractall(work)
    else:
        manifest_candidates = sorted(Path('/kaggle/input').glob('**/bundle_manifest.json'))
        if not manifest_candidates:
            raise FileNotFoundError('Attach axon_phase0b_kaggle_bundle as a Kaggle dataset first.')
        source_root = manifest_candidates[0].parent
        print('expanded bundle dir:', source_root)
        shutil.copytree(source_root, work, dirs_exist_ok=True)

manifest = json.loads((work / 'bundle_manifest.json').read_text())
print(json.dumps(manifest['checkpoint'], indent=2))
print('phase0b families:', manifest['phase0b']['families'])

In [ ]:
try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi not found')

repo = work / 'axon'
checkpoint_meta = manifest['checkpoint']
training = manifest['training']
checkpoint = work / checkpoint_meta['archive_path']
phase0b_rel = Path(manifest['phase0b'].get('archive_relative', 'datasets/recovered/phase0b_curriculum_v1'))
if phase0b_rel.is_absolute() or '..' in phase0b_rel.parts:
    raise RuntimeError(f'unsafe phase0b archive path: {phase0b_rel}')
phase0b_dir = repo / phase0b_rel
assert repo.exists(), repo
assert checkpoint.exists(), checkpoint
assert phase0b_dir.exists(), phase0b_dir

start_step = int(training['start_step'])
target_step = int(training['target_step'])
leg_steps = int(training['leg_steps'])
if int(checkpoint_meta['start_step']) != start_step:
    raise RuntimeError('checkpoint and training start_step disagree')
if target_step <= start_step or leg_steps != target_step - start_step:
    raise RuntimeError(f'invalid leg contract: {start_step} -> {target_step}, leg_steps={leg_steps}')
if checkpoint.stat().st_size != int(checkpoint_meta['bytes']):
    raise RuntimeError('source checkpoint byte-size mismatch')

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

verified_file_sha256 = {}
for entry in manifest.get('files', []):
    relative = Path(entry['path'])
    if relative.is_absolute() or '..' in relative.parts:
        raise RuntimeError(f'unsafe bundle file path: {relative}')
    bundled_file = work / relative
    if not bundled_file.is_file() or bundled_file.stat().st_size != int(entry['bytes']):
        raise RuntimeError(f'bundle file size mismatch: {relative}')
    actual_sha256 = sha256_file(bundled_file)
    if actual_sha256 != entry['sha256']:
        raise RuntimeError(f'bundle file SHA-256 mismatch: {relative}')
    verified_file_sha256[relative.as_posix()] = actual_sha256
bundle_manifest_sha256 = sha256_file(work / 'bundle_manifest.json')
checkpoint_sha256 = verified_file_sha256.get(checkpoint_meta['archive_path'])
if checkpoint_sha256 is None:
    checkpoint_sha256 = sha256_file(checkpoint)
if checkpoint_sha256 != checkpoint_meta['sha256']:
    raise RuntimeError('source checkpoint SHA-256 mismatch')
curriculum_manifest_path = phase0b_dir / 'phase0b_exact_curriculum_manifest.json'
curriculum_manifest = json.loads(curriculum_manifest_path.read_text())
curriculum_manifest_sha256 = sha256_file(curriculum_manifest_path)
if curriculum_manifest_sha256 != manifest['phase0b']['manifest_sha256']:
    raise RuntimeError('exact-v3 curriculum manifest SHA-256 mismatch')
if curriculum_manifest.get('schema') != manifest['phase0b']['manifest_schema']:
    raise RuntimeError('exact-v3 curriculum schema mismatch')
if curriculum_manifest.get('schema') != 'axon_phase0b_exact_curriculum_manifest_v3':
    raise RuntimeError('quarantined pilot requires exact-v3 curriculum')
family_list = list(manifest['phase0b']['families'])
if set(curriculum_manifest.get('families', {})) != set(family_list):
    raise RuntimeError('exact-v3 family set mismatch')
sampler_contract = curriculum_manifest.get('training_sampler_contract', {})
if sampler_contract.get('schema') != 'axon_weighted_family_sampler_v1' or sampler_contract.get('required') is not True:
    raise RuntimeError('exact-v3 weighted sampler contract is missing')
curriculum_audit = {}
for family in family_list:
    family_path = phase0b_dir / f'{family}.jsonl'
    split_counts = {'train': 0, 'dev': 0, 'test': 0}
    allowed_counts = {}
    for line in family_path.open(encoding='utf-8'):
        row = json.loads(line)
        if row.get('schema') != 'axon_phase0b_delta_v3':
            raise RuntimeError(f'non-v3 curriculum row in {family}')
        audit = row.get('budget_audit', {})
        if audit.get('silent_clips') != 0 or audit.get('unsupported_substitutions') != 0:
            raise RuntimeError(f'lossy curriculum row in {family}')
        modes = tuple(row.get('allowed_draft_modes', []))
        if not modes or any(mode not in {'copy', 'partial', 'blank'} for mode in modes):
            raise RuntimeError(f'invalid allowed_draft_modes in {family}')
        split_counts[row['split']] += 1
        key = ','.join(modes)
        allowed_counts[key] = allowed_counts.get(key, 0) + 1
    expected_counts = curriculum_manifest['families'][family]['counts']
    if split_counts != expected_counts:
        raise RuntimeError(f'curriculum split counts mismatch for {family}: {split_counts} != {expected_counts}')
    selected_audit = curriculum_manifest['families'][family].get('selected_audit', {})
    for zero_key in ('avoidable_midword_boundaries', 'broken_chains', 'midword_boundaries', 'silent_clips', 'unsupported_substitutions'):
        if selected_audit.get(zero_key) != 0:
            raise RuntimeError(f'exact-v3 selected audit failed: {family}.{zero_key}')
    curriculum_audit[family] = {
        'split_counts': split_counts,
        'allowed_draft_modes': allowed_counts,
        'selected_audit': selected_audit,
    }
print('repo:', repo)
print('checkpoint:', checkpoint)
print('checkpoint sha256:', checkpoint_sha256)
print(f'verified bundle files: {len(verified_file_sha256)} manifest_sha256={bundle_manifest_sha256}')
print('phase0b_dir:', phase0b_dir)
print('curriculum manifest sha256:', curriculum_manifest_sha256)
print('curriculum audit:', json.dumps(curriculum_audit, indent=2))
print(f'verified leg: {start_step} -> {target_step} ({leg_steps} steps)')

In [ ]:
import torch
sys.path.insert(0, str(repo))
from substrate import SLOT_DIM, default_alphabet
from training.checkpoint_contract import validate_checkpoint_continuity
from training.trainer_slot import CharSlotFieldBuilder, run_charslot_roundtrip_gate

if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is required for this training leg')
device = 'cuda'
run_name = str(training.get('run_name', 'coreA64_phase0b'))
core_cfg = str(training.get('core_cfg', 'A'))
run_dir = Path('/kaggle/working/runs') / run_name
family_list = list(manifest['phase0b']['families'])
source_family_list = list(checkpoint_meta.get('curriculum_families', family_list))
families = ','.join(family_list)
mode_weights = str(training.get('charslot_mode_weights', '0.3,0.3,0.4'))
partial_frac = float(training.get('charslot_partial_frac', 0.5))
eval_partial_frac = float(training.get('eval_partial_frac', 0.5))
learning_rate = float(training['learning_rate'])
resume_lr_policy = str(training['resume_lr_policy'])
resume_curriculum_state_policy = str(training['resume_curriculum_state_policy'])
curriculum_reset_reason = str(training.get('curriculum_reset_reason', ''))
family_sampling = str(training.get('phase0b_family_sampling', 'global'))
family_weights = list(training.get('phase0b_family_weights', []))
family_weight_arg = ','.join(str(value) for value in family_weights)
eval_every = int(training.get('eval_every', 500))
eval_n = int(training.get('eval_n', 64))
checkpoint_every = int(training.get('checkpoint_every', 1000))
seed = int(training.get('seed', 0))
leg_kind = str(training.get('leg_kind', 'standard'))
leg_gate = training.get('leg_gate')
gate_role = training.get('gate_role')
evaluation_contract = training['evaluation_contract']
if not isinstance(leg_gate, dict):
    raise RuntimeError('quarantined pilot leg gate is missing')
continuity_contract = training['continuity']
if leg_gate.get('fixed_suite_sha256') != evaluation_contract['fixed_suite_sha256']:
    raise RuntimeError('pilot gate and fixed evaluation suite disagree')
source_payload = torch.load(checkpoint, map_location='cpu', weights_only=False, mmap=True)
source_checkpoint_contract = validate_checkpoint_continuity(
    source_payload, expected_step=start_step, expected_families=source_family_list, require_cuda_rng=True
)
if not source_checkpoint_contract['valid']:
    raise RuntimeError(f'source checkpoint continuity failed: {source_checkpoint_contract["violations"]}')
source_core_config = source_payload.get('cfg')
del source_payload
roundtrip_builder = CharSlotFieldBuilder(256, 64, 64, torch.device('cpu'), torch.float32)
supported_text = ''.join(default_alphabet())
if SLOT_DIM != 16 or len(supported_text) != 95:
    raise RuntimeError(f'invalid frozen substrate gate: slot_dim={SLOT_DIM} chars={len(supported_text)}')
run_charslot_roundtrip_gate(roundtrip_builder)
roundtrip_gate = {
    'passed': True,
    'slot_dim': SLOT_DIM,
    'supported_characters': len(supported_text),
    'alphabet_sha256': hashlib.sha256(supported_text.encode('utf-8')).hexdigest(),
}

cmd = [
    sys.executable, 'training/trainer_slot.py',
    '--mode', 'phase0b',
    '--threshold', 'charslot',
    '--resume', str(checkpoint),
    '--run-dir', str(run_dir),
    '--core-cfg', core_cfg,
    '--device', device,
    '--steps', str(target_step),
    '--phase0b-dir', str(phase0b_dir),
    '--phase0b-families', families,
    '--charslot-mode-weights', mode_weights,
    '--charslot-partial-frac', str(partial_frac),
    '--eval-partial-frac', str(eval_partial_frac),
    '--charslot-rung-preset', 'manual',
    '--phase0b-family-sampling', family_sampling,
    '--phase0b-family-weights', family_weight_arg,
    '--lr', str(learning_rate),
    '--resume-lr-policy', resume_lr_policy,
    '--resume-curriculum-state-policy', resume_curriculum_state_policy,
    '--resume-curriculum-reset-reason', curriculum_reset_reason,
    '--history-chars', '256',
    '--user-chars', '64',
    '--max-response-chars', '64',
    '--eval-every', str(eval_every),
    '--eval-n', str(eval_n),
    '--eval-samples', '3',
    '--checkpoint-every', str(checkpoint_every),
    '--log-every', '50',
    '--seed', str(seed),
    '--eval-source-on-resume',
    '--grad-checkpoint',
]
print('roundtrip gate:', json.dumps(roundtrip_gate, indent=2))
print(f'leg_kind={leg_kind} gate_role={gate_role} gate={bool(leg_gate)}')
print(' '.join(cmd))
subprocess.run(cmd, cwd=repo, check=True)

In [ ]:
out_base = Path('/kaggle/working/axon_phase0b_outputs')
final_pointer = run_dir / 'pointer.json'
final_done = run_dir / 'checkpoint_done.json'
source_eval_file = run_dir / 'eval_source.json'
final_eval = run_dir / 'eval_final.json'
if not final_pointer.exists() or not final_done.exists() or not source_eval_file.exists() or not final_eval.exists():
    raise RuntimeError('trainer did not persist checkpoint sentinels')
pointer = json.loads(final_pointer.read_text())
done = json.loads(final_done.read_text())
final_step = int(pointer['step'])
if int(done['step']) != final_step:
    raise RuntimeError(f'checkpoint sentinels disagree: pointer={final_step} done={done.get("step")}')
completed = final_step == target_step
if not completed:
    print(f'QUARANTINE: expected persisted step {target_step}, got {final_step}')
active_checkpoint = run_dir / pointer['active']
if not active_checkpoint.is_file() or active_checkpoint.stat().st_size == 0:
    raise RuntimeError(f'missing active checkpoint: {active_checkpoint}')
active_sha256 = sha256_file(active_checkpoint)
final_payload = torch.load(active_checkpoint, map_location='cpu', weights_only=False, mmap=True)
if int(final_payload.get('step', -1)) != final_step:
    raise RuntimeError(f'final checkpoint payload step mismatch: {final_payload.get("step")}')
final_checkpoint_contract = validate_checkpoint_continuity(
    final_payload,
    expected_step=final_step,
    expected_families=family_list,
    require_cuda_rng=True,
    expected_source_step=start_step,
    expected_optimizer_policy=continuity_contract['expected_optimizer_policy'],
    expected_optimizer_state_restored=continuity_contract['expected_optimizer_state_restored'],
    expected_effective_lr=continuity_contract['expected_effective_lr'],
    expected_sampler_policy=continuity_contract['expected_sampler_policy'],
    expected_sampler_reset=continuity_contract['expected_sampler_reset'],
)
if not bool(done.get('optimizer_state')):
    final_checkpoint_contract['valid'] = False
    final_checkpoint_contract['violations'].append('checkpoint_done does not attest optimizer state')
if final_payload.get('cfg') != source_core_config:
    final_checkpoint_contract['valid'] = False
    final_checkpoint_contract['violations'].append('core configuration changed across repair leg')
del final_payload
source_eval_result = json.loads(source_eval_file.read_text())
eval_result = json.loads(final_eval.read_text())
if int(source_eval_result.get('step', -1)) != start_step:
    raise RuntimeError('source evaluation step does not match source checkpoint')
if int(eval_result.get('step', -1)) != final_step or int(eval_result.get('eval_step', -1)) != final_step:
    raise RuntimeError('final evaluation step does not match checkpoint')
if source_eval_result.get('fixed_suite_sha256') != evaluation_contract['fixed_suite_sha256']:
    raise RuntimeError('source evaluation suite SHA-256 mismatch')
if eval_result.get('fixed_suite_sha256') != evaluation_contract['fixed_suite_sha256']:
    raise RuntimeError('final evaluation suite SHA-256 mismatch')
if source_eval_result.get('family_counts') != evaluation_contract['family_counts']:
    raise RuntimeError('source evaluation family counts mismatch')
if eval_result.get('family_counts') != evaluation_contract['family_counts']:
    raise RuntimeError('final evaluation family counts mismatch')
gate_result = {'passed': None, 'continuable': False, 'promotable': False, 'violations': []}
if leg_gate:
    if completed:
        sys.path.insert(0, str(repo))
        if leg_gate.get('schema') == 'axon_charslot_pilot_gate_v1':
            from training.promotion_gate import evaluate_pilot_gate
            gate_result = evaluate_pilot_gate(source_eval_result, eval_result, leg_gate)
        else:
            from training.promotion_gate import evaluate_promotion_gate
            legacy_result = evaluate_promotion_gate(eval_result.get('metrics', {}), leg_gate)
            gate_result = {**legacy_result, 'continuable': legacy_result['passed'], 'promotable': legacy_result['passed']}
    else:
        gate_result = {'passed': False, 'continuable': False, 'promotable': False, 'violations': [f'incomplete leg: {final_step} != {target_step}']}
    print('LEG GATE:', json.dumps(gate_result, indent=2))
continuable = bool(completed and final_checkpoint_contract['valid'] and gate_result['continuable'] is True)
promotable = bool(completed and final_checkpoint_contract['valid'] and gate_result['promotable'] is True)
leg_result = {
    'schema': 'axon_training_leg_result_v3',
    'run_name': run_name,
    'core_cfg': core_cfg,
    'leg_kind': leg_kind,
    'charslot_mode_weights': mode_weights,
    'charslot_partial_frac': partial_frac,
    'eval_partial_frac': eval_partial_frac,
    'learning_rate': learning_rate,
    'resume_lr_policy': resume_lr_policy,
    'resume_curriculum_state_policy': resume_curriculum_state_policy,
    'curriculum_reset_reason': curriculum_reset_reason,
    'phase0b_family_sampling': family_sampling,
    'phase0b_family_weights': family_weights,
    'phase0b_families': family_list,
    'start_step': start_step,
    'target_step': target_step,
    'leg_steps': leg_steps,
    'final_step': final_step,
    'completed': completed,
    'optimizer_state': final_checkpoint_contract['optimizer']['valid'],
    'rng_state': final_checkpoint_contract['rng']['valid'],
    'curriculum_state': final_checkpoint_contract['sampler']['valid'],
    'source_checkpoint_sha256': checkpoint_sha256,
    'active_checkpoint': pointer['active'],
    'active_checkpoint_bytes': active_checkpoint.stat().st_size,
    'active_checkpoint_sha256': active_sha256,
    'bundle_manifest_sha256': bundle_manifest_sha256,
    'verified_bundle_file_count': len(verified_file_sha256),
    'roundtrip_gate': roundtrip_gate,
    'curriculum_manifest_sha256': curriculum_manifest_sha256,
    'curriculum_audit': curriculum_audit,
    'source_checkpoint_contract': source_checkpoint_contract,
    'final_checkpoint_contract': final_checkpoint_contract,
    'source_eval': source_eval_result,
    'eval': eval_result,
    'gate_role': gate_role,
    'leg_gate': leg_gate,
    'gate_result': gate_result,
    'continuation_gate_passed': gate_result['continuable'],
    'promotion_gate_passed': gate_result['promotable'],
    'gate_violations': gate_result['violations'],
    'continuable': continuable,
    'promotable': promotable,
}
leg_manifest_path = run_dir / 'leg_result.json'
leg_manifest_path.write_text(json.dumps(leg_result, indent=2) + '\n')
Path('/kaggle/working/axon_phase0b_leg_result.json').write_text(json.dumps(leg_result, indent=2) + '\n')
archive = shutil.make_archive(str(out_base), 'zip', '/kaggle/working/runs')
if not Path(archive).is_file():
    raise RuntimeError('output archive was not created')
print(json.dumps(leg_result, indent=2))
print('outputs:', archive)
print('latest run files:')
for path in sorted(run_dir.glob('*')):
    print(path, path.stat().st_size)